# 실제 금리 모형의 양의 관계도 여섯 관측치에 민감할까?

02번에서 예상과 반대 방향의 관계가 보였다. 03번과 정확히 같은 업종·연도 여섯 행을 빼면 그 관계가 얼마나 남는지 확인한다.

ICR(이자보상배율)은 영업이익을 이자비용으로 나눈 값이다. 3배라면 영업이익이 그해 이자비용의 세 배라는 뜻이다. 한 행은 한 업종의 한 해다. 


[전체 질문 목록](../README.md)

**표 읽는 법**　N은 업종·연도 관측 수, G는 업종 수다. 회귀표의 beta는 추정한 관계의 크기, lo와 hi는 95% 신뢰구간의 양 끝이다. p값은 계수가 0이라는 가정에서 지금만큼 또는 더 극단적인 검정통계량이 나올 확률이다.

# 실제 기준금리 모형 + 동일 극단값 제외
2026-11-02 사용자 승인. 기존 자료와 노트북은 보존. 별도 보조 민감도 분석이다.

## 고정 사양 및 변경 기록
기존 실제 금리 C/D 모형의 표본만 240행에서 전체기간 ICR 1/99% 경계 밖 행을 제외한 표본으로 변경한다. 이전 기간모형의 제외 관측치와 정확히 같은지 검증한다. 1%는 오류 판정 공식 기준이 아닌 기존 비교에 맞춘 분석자 선택이며, 회귀 결과에 따라 변경하지 않는다.

2018–23년 제조업 40업종. Y=원자료 ICR%/100(배). E=2015–17년의 100×(단기차입+단기사채)/총자산 연도별 비율 평균; L=동 기간 100×유동자산/유동부채 평균. E/L은 제외 이후 재계산하지 않는다. r=기준금리 변경 당일부터 적용한 일별 금리의 연평균(%). 원 금리 이력 CSV를 그대로 사용한다.

C: Y_it=업종FE+연도FE+β(E_i×r_t)+오차.
D: C+γ(L_i×r_t). D가 이번 대안 모형의 중심이다. 기간모형 M1–M5를 재실행한 것이 아니며, 원자재·환율·결합항·시차·금리차분을 추가하지 않는다.

β: 사전 단기차입 의존도 1%p 차이 및 금리 1%p 차이에 따른 상대 ICR 차이(배). 금리 단독항은 연도FE에 흡수되므로 금리의 전체 인과효과는 추정하지 않는다. CR1 업종 군집 SE 및 t(G−1) 95% CI; WCR-S31/C11 9,999회, Rademacher, seed=20261102. 모든 p값은 양측·다중비교 미조정. 절단 경계와 표본을 고정한 조건부 추론이다.

재실행: Python3.12 및 requirements.txt 환경, 노트북 폴더를 작업경로로 전체 셀 실행. inputs 해시 검증 후 재계산한다. 원자료에서 최초 병합패널 구축은 기존 01_analysis.ipynb가 담당하고, 이 노트북은 동결 패널부터 금리·사전 재무구조·제외 표본·회귀를 재현한다.


In [1]:
# 금리와 사전 노출 준비
from pathlib import Path
import json,hashlib
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from IPython.display import display
from wildboottest.wildboottest import wildboottest
BASE = Path.cwd()
(BASE / 'tables').mkdir(exist_ok=True)
for x in json.loads((BASE/'inputs/manifest.json').read_text(encoding='utf-8')):
 assert hashlib.sha256((BASE/'inputs'/x['file']).read_bytes()).hexdigest()==x['sha256']
def save(d, n):
    d.to_csv(BASE / 'tables' / f'{n}.csv', index=False, encoding='utf-8-sig')
d = pd.read_csv(BASE / 'inputs/panel.csv')
s = pd.read_csv(BASE / 'inputs/stability.csv')
assert len(d)==240 and d.code.nunique()==40 and set(d.year)==set(range(2018,2024))
assert not d.duplicated(['code','year']).any()
s['Enew'] = 100 * (s['단기차입'] + s['단기사채']) / s['자산총계']
s['Lnew'] = 100 * s['유동자산'] / s['유동부채']
pre = s[s['연도'].between(2015, 2017)]
assert pre.groupby('업종코드').size().eq(3).all()
av = pre.groupby('업종코드')[['Enew', 'Lnew']].mean()
d['Enew'] = d.code.map(av.Enew)
d['Lnew'] = d.code.map(av.Lnew)
rr = pd.read_csv(BASE / 'inputs/rates.csv')
rr['date'] = pd.to_datetime(rr['변경일자'])
rr = rr.sort_values('date')
assert not rr.date.duplicated().any()
# 이 분석에 필요한 2018–23만 산출. 2024년 이후 기록은 사용하지 않는다.
calendar=pd.DataFrame({'date':pd.date_range('2018-01-01','2023-12-31')})
daily = pd.merge_asof(calendar, rr[['date', '기준금리']], on='date', direction='backward')
assert daily['기준금리'].notna().all()
annual=daily.groupby(daily.date.dt.year)['기준금리'].mean().rename('rate').rename_axis('year').reset_index()
d['rate']=d.year.map(annual.set_index('year').rate)
for v in ['E', 'L']:
    d[v + 'P_new'] = d[v + 'new'] * d.Post
    d[v + 'T_new'] = d[v + 'new'] * (d.year == 2021)
    d[v + 'R'] = d[v + 'new'] * d.rate
assert np.isfinite(d[['Y','Enew','Lnew','rate','ER','LR']]).all().all()
save(annual, 'annual_rates')
save(d, 'analysis_panel')
display(annual)
print('업종별 사전 E/L:',len(av),'금리 연도:',len(annual))

# 제외된 업종과 연도 대조
full = d.copy()
low, high = full.Y.quantile([0.01, 0.99], interpolation='linear')
excluded = full.loc[~full.Y.between(low, high)].copy()
d = full.loc[full.Y.between(low, high)].copy().reset_index(drop=True)
ref=pd.read_csv(BASE/'inputs/excluded_reference.csv')
assert set(zip(excluded.code,excluded.year))==set(zip(ref.code,ref.year))
assert len(d)==234 and d.code.nunique()==40
save(excluded, 'excluded_observations')
save(d, 'trimmed_panel')
save(pd.DataFrame([{'lower':low,'upper':high,'N_full':len(full),'N_trim':len(d),'G':d.code.nunique()}]),'sample_audit')
print('Same exclusion boundaries:', low, high)
display(excluded[['industry', 'year', 'Y']])


   year      rate
0  2018  1.521918
1  2019  1.582877
2  2020  0.704918
3  2021  0.613014
4  2022  2.032192
5  2023  3.491781
업종별 사전 E/L: 40 금리 연도: 6
Same exclusion boundaries: -7.2226300000000005 82.87627999999987
    industry  year        Y
19       컴퓨터  2019   -9.300
29      통신기기  2023   -9.630
177       조선  2021   -7.372
188       담배  2020   86.444
189       담배  2021  129.613
190       담배  2022  134.803


### 출력에서 읽을 수 있는 것

위쪽 표는 여섯 연도의 금리, 아래쪽 표는 제외된 여섯 업종·연도다. 같은 숫자 6이지만 서로 다른 대상을 센다.

- 원래 금리 분석과 같은 사전 노출과 연평균 금리를 만들었다. 이렇게 해야 다음 비교에서 바뀌는 요인을 표본으로 한정할 수 있다.
- 기간 모형에서 제외한 업종·연도 목록과 정확히 같은 6행임을 확인했다. 금리 모형 비교에 남은 표본은 234행이다.

In [2]:
# 공통 추정식 정의
def design(d,terms):
    X=pd.concat([pd.DataFrame({'const':np.ones(len(d))},index=d.index),d[terms],pd.get_dummies(d.code,prefix='i',drop_first=True,dtype=float),pd.get_dummies(d.year,prefix='y',drop_first=True,dtype=float)],axis=1).astype(float)
    assert np.linalg.matrix_rank(X)==X.shape[1]
    return X
fits = {}
rows = []
def fit(label, d, terms, y='Y'):
    X = design(d, terms)
    model = sm.OLS(d[y].astype(float), X)
    res = model.fit(cov_type='cluster', cov_kwds={'groups': d.code, 'use_correction': True, 'df_correction': True}, use_t=True)
    G = d.code.nunique()
    q = stats.t.ppf(0.975, G - 1)
    for term in terms:
        b = float(res.params[term])
        se = float(res.bse[term])
        rows.append(dict(model=label, outcome=y, term=term, beta=b, se=se, lo=b - q * se, hi=b + q * se, p=float(2 * stats.t.sf(abs(b / se), G - 1)), N=len(d), G=G))
    fits[label] = (d.copy(), terms, model, res)
    return res

# 두 표본에서 금리 모형 추정
for sample, dd in [('full', full), ('trim', d)]:
    fit(sample + '_C', dd, ['ER'])
    fit(sample + '_D', dd, ['ER', 'LR'])
results = pd.DataFrame(rows)
save(results, 'coefficients')
display(results)

# 기존 결과와 검산
ref=pd.read_csv(BASE/'inputs/original_coefficients.csv')
for name, oldname in [('C', 'C_rate_E'), ('D', 'D_rate_EL')]:
    a = results[results.model == 'full_' + name].set_index('term')
    b = ref[ref.model == oldname].set_index('term')
    np.testing.assert_allclose(a.loc[b.index, ['beta', 'se', 'p']], b[['beta', 'se', 'p']], atol=1e-08)
X = design(d, ['ER', 'LR'])
assert np.linalg.matrix_rank(np.column_stack([X, d.rate])) == X.shape[1]
model = fits['trim_D'][2]
res = fits['trim_D'][3]
X = model.exog
y = model.endog
b = np.linalg.lstsq(X, y, rcond=None)[0]
u = y - X @ b
bread = np.linalg.inv(X.T @ X)
sc = np.array([X[d.code == g].T @ u[d.code == g] for g in sorted(d.code.unique())])
G = d.code.nunique()
N = len(d)
cov=bread@sc.T@sc@bread*G/(G-1)*(N-1)/(N-X.shape[1])
np.testing.assert_allclose(b, res.params, atol=1e-08)
np.testing.assert_allclose(cov, res.cov_params(), atol=1e-07)
print('Original C/D reproduced; trimmed D OLS/CR1 independently checked')


    model outcome term      beta  ...        hi         p    N   G
0  full_C       Y   ER  0.432142  ...  0.843973  0.040204  240  40
1  full_D       Y   ER  0.327311  ...  0.699104  0.082753  240  40
2  full_D       Y   LR -0.027096  ...  0.021843  0.269607  240  40
3  trim_C       Y   ER  0.182864  ...  0.412471  0.115263  234  40
4  trim_D       Y   ER  0.180226  ...  0.439649  0.167879  234  40
5  trim_D       Y   LR -0.000724  ...  0.021678  0.948225  234  40

[6 rows x 10 columns]
Original C/D reproduced; trimmed D OLS/CR1 independently checked


### 출력에서 읽을 수 있는 것

full_C의 p=0.040에서 trim_C는 p=0.115로 바뀌고 계수도 작아졌다. 큰 관측치의 처리에 따라 양의 관계에 대한 판단이 달라질 수 있다.

- 두 표본에서 금리 모형의 계수를 같은 방식으로 계산할 함수를 정의했다.
- 6행 제외 후 C는 +0.183(p=0.115), D는 +0.180(p=0.168)이었다. 전체표본 C의 +0.432보다 추정 크기가 작아졌다.
- 전체표본 결과가 이전 C·D와 일치하고, 제한표본 D의 계수와 표준오차가 별도 계산과도 맞는지 확인했다.

In [3]:
# 제한표본 재검정
boot=[]
for label, terms in [('trim_C', ['ER']), ('trim_D', ['ER', 'LR'])]:
    dd, _, model, res = fits[label]
    xx = pd.DataFrame(model.exog, columns=model.exog_names)
    yy = pd.Series(model.endog)
    gg = pd.Series(dd.code.to_numpy())
    xx = xx - xx.groupby(gg).transform('mean')
    yy = yy - yy.groupby(gg).transform('mean')
    xx = xx.loc[:, np.max(abs(xx), axis=0) > 1e-10]
    assert np.linalg.matrix_rank(xx) == xx.shape[1]
    mm = sm.OLS(yy, xx)
    for term in terms:
        np.testing.assert_allclose(mm.fit().params[term], res.params[term], atol=1e-09)
        for variant in ['31', '11']:
            w = wildboottest(mm, param=term, cluster=pd.factorize(gg)[0], B=9999, seed=20261102, bootstrap_type=variant, impose_null=True, weights_type='rademacher', show=False)
            boot.append({'model': label, 'term': term, 'variant': variant, 'p_boot': float(w['p-value'].iloc[0]), 'B': 9999, 'seed': 20261102})
boot = pd.DataFrame(boot)
save(boot, 'bootstrap')
display(boot)


    model term variant    p_boot     B      seed
0  trim_C   ER      31  0.093109  9999  20261102
1  trim_C   ER      11  0.093109  9999  20261102
2  trim_D   ER      31  0.158816  9999  20261102
3  trim_D   ER      11  0.159816  9999  20261102
4  trim_D   LR      31  0.940394  9999  20261102
5  trim_D   LR      11  0.942394  9999  20261102


### 출력에서 읽을 수 있는 것

trim_C의 ER은 p_boot=0.093, trim_D의 ER은 0.159다. 유동성까지 넣은 행과 단기차입만 넣은 행을 구분한다.

## 근거와 한계
[NIST/SEMATECH e-Handbook §1.3.5.17, Introduction](https://itl.nist.gov/div898/handbook/eda/section3/eda35h.htm)의 관련 본문을 앞선 검토에서 확인했다. “we typically do not want to simply delete the outlying observation”. 오류가 확인되지 않은 극단값을 자동 삭제하지 말라는 근거이며 1% 절단을 정당화하는 문헌이 아니다. 제외 전후 비교를 함께 보고하고 원본을 보존한다.

기준금리와 기업의 실제 부담금리는 다르다. 기존 금리모형 보고서의 한국은행 이력 검증 및 Fed(2023) 관련 본문 확인을 따른다. 업종 간 오차 독립성, 6개뿐인 연간 금리 관측, 선형·동시점 관계 가정, 업종별 경기 누락, 오래된 사전 재무구조와 산업 집계 정의 미확인 한계는 남는다. Y로 표본을 제한하므로 선택에 따른 왜곡 가능성이 있고, 제외 결과가 더 정확하다거나 인과적이라고 주장할 수 없다. 연구 질문을 지지하는 결과를 얻기 위해 경계를 바꾸지 않는다.


# 실제 금리 × 극단값 제외: 결과

결론: 앞서 실제 금리 모형에서 관찰된 단기차입의 양의 관계는 같은 극단값 제외 기준을 적용하면 약해졌다. 단기차입만 넣은 C에서도 관례적 5% 유의성이 유지되지 않았다. 기존 취약성 가설의 음의 방향은 이번에도 확인되지 않았다. 관계가 정확히 0이라는 결론은 아니다.

## 무엇을 바꿨나
실제 금리 모형 C/D의 2018–23년, 2015–17년 사전 재무구조 평균, 일수 가중 연평균 기준금리, 업종·연도 고정효과는 모두 유지했다. 전체기간 ICR의 하위 1%·상위 1% 밖 6개 관측치만 제외했다. 경계는 -7.222630배와 82.876280배로, 직전 기간모형 분석과 정확히 같다. 표본은 240→234행이며 40업종은 모두 남는다. 제외 후 사전 재무구조나 금리 값을 재추정하지 않았다.

제외: 컴퓨터 2019(-9.300), 통신기기 2023(-9.630), 조선 2021(-7.372), 담배 2020(86.444)·2021(129.613)·2022(134.803). 특정 업종 전체를 제외한 것이 아니고, 해당 값의 오류를 확인한 것도 아니다. 1%는 이전 분석과 비교하기 위한 분석자 선택이며 공식 오류 기준이 아니다.

## 핵심 결과
아래 표는 단기차입 의존도×금리 항이다. p값은 업종 군집 CR1 및 t(39), CI는 같은 방식의 95% 구간이다.

| 모형 | 원본 계수 (p값) | 제외 후 계수 | 제외 후 95% CI | 제외 후 p값 |
|---|---|---:|---|---:|
| C: 단기차입만 | +0.4321 (0.0402) | +0.1829 | [-0.0467, 0.4125] | 0.1153 |
| D: 단기차입+유동비율 | +0.3273 (0.0828) | +0.1802 | [-0.0792, 0.4396] | 0.1679 |

D의 유동비율×금리 계수는 -0.000724, 95% CI [-0.02313, 0.02168], p=0.9482다. 완충 결합항을 추정한 모형은 아니다.

업종 군집 부트스트랩 WCR-S의 p값은 C 단기차입 0.0931, D 단기차입 0.1588, D 유동비율 0.9404이다. WCR-C도 아래 전체 표에 공개한다. 9,999회, seed=20261102; 다중비교 미조정이며 선택된 표본과 경계를 고정한 조건부 검정이다.

## 어떻게 해석하나
계수는 사전 단기차입 의존도 1%p 차이 및 금리 1%p 차이에 따른 상대 ICR 차이(배)다. 업종에 공통인 금리 자체의 평균 효과는 연도 고정효과와 구분해 추정하지 못한다.

원본 C에서 보였던 양의 관계는 이번 표본 제한에 민감하다. 따라서 단기차입이 금리 상승 때 보호 역할을 한다는 결론을 강화하지 않는다. 반대로 제외 후 결과가 정답이거나 원본 결과가 오류라는 뜻도 아니다. 제외한 6개 행을 묶어서 비교했으므로 어느 한 업종이 원인이라고 특정하지 않는다. 제외 전후 계수 차이 자체의 유의성을 검정한 것은 아니다.

원래 가설인 단기차입 취약성의 음의 관계를 확인하지 못했다는 해석은 유지된다. 전체 표본을 보존하고 이번 결과를 민감도 분석으로 함께 제시하는 것을 권고한다. 이 결과를 이유로 제외 폭을 늘리거나 시차·기간·변수를 자동으로 바꾸지 않는다.

## 근거와 한계
[NIST/SEMATECH e-Handbook §1.3.5.17 Introduction](https://itl.nist.gov/div898/handbook/eda/section3/eda35h.htm)의 관련 본문을 앞선 검토에서 확인: “we typically do not want to simply delete the outlying observation”. 오류가 확인되지 않은 값을 단순 삭제하지 말라는 근거로 사용한다. 해당 지침이 1% 절단을 공식 기준으로 정한 것은 아니다.

종속변수로 표본을 제한하므로 실제 취약성이나 높은 상환능력 관측까지 제외할 수 있다. 연간 금리 관측은 여전히 6개다. 업종 간 잔여 오차 독립성·금리의 선형 동시점 관계·업종 경기 누락 등의 기존 한계는 해결되지 않았다. 기존 보고서의 기준금리 이력 검증과 사전 재무구조 정의를 그대로 적용했다.

## 재현 확인
전체 표본 C/D의 계수·표준오차·p값을 기존 결과와 대조해 재현했다. 제외 관측치가 직전 분석과 동일함을 확인했다. 제외 후 D의 OLS 및 CR1 공분산을 별도 행렬 공식으로 검산했다. 원본 및 복사 입력의 해시를 확인했다. 3개 코드 셀을 동일 Python 실행 공간에서 순서대로 실행하고 출력을 저장했다. 노트북과 inputs·requirements.txt를 함께 사용한다.
